In [ ]:
import os
import re
import subprocess
from datetime import datetime
import nbformat
from nbconvert import HTMLExporter

# --- CONFIGURATION ---
GITHUB_USERNAME = "adte-net"
REPO_NAME = "IDS-Development"

# Always specify paths RELATIVE TO YOUR REPOSITORY ROOT:
NOTEBOOK_REPO_REL_PATH = "Chunk01/02b_zone_policy_explorer.ipynb"
HTML_FILENAME = "02b_zone_policy_explorer.html"
# ---------------------

# Automatically detect the git repository root directory
try:
    repo_root = subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()
except Exception as e:
    raise RuntimeError("Could not determine git repository root. Make sure you are inside a git repo.")

notebook_abs_path = os.path.join(repo_root, NOTEBOOK_REPO_REL_PATH)
docs_dir_abs = os.path.join(repo_root, "docs")
dest_path = os.path.join(docs_dir_abs, HTML_FILENAME)
os.makedirs(docs_dir_abs, exist_ok=True)

if not os.path.exists(notebook_abs_path):
    print(f"❌ ERROR: Could not find notebook at: {notebook_abs_path}")
else:
    print(f"1. Converting {NOTEBOOK_REPO_REL_PATH} to HTML...")
    
    # Convert using Python API (avoids CLI path/quoting bugs)
    with open(notebook_abs_path, "r", encoding="utf-8") as f:
        notebook_node = nbformat.read(f, as_version=4)
    
    html_exporter = HTMLExporter()
    body, _ = html_exporter.from_notebook_node(notebook_node)
    
    with open(dest_path, "w", encoding="utf-8") as f:
        f.write(body)

    # --- INJECT HEADER & ANTI-CACHE TAGS ---
    print("Injecting header banner and anti-cache headers into HTML...")
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    base_notebook_name = os.path.basename(NOTEBOOK_REPO_REL_PATH)

    header_banner = f"""
    <div style="background-color: #f6f8fa; border-bottom: 1px solid #d1d5db; padding: 12px 20px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 14px; color: #24292e; display: flex; justify-content: space-between; align-items: center; box-shadow: 0 1px 3px rgba(0,0,0,0.05); margin-bottom: 20px;">
        <div><strong>Notebook:</strong> {base_notebook_name}</div>
        <div><strong>Last Generated:</strong> {current_time}</div>
    </div>
    """

    anti_cache_tags = """
    <!-- Anti-Cache Injection -->
    <meta http-equiv="Cache-Control" content="no-cache, no-store, must-revalidate">
    <meta http-equiv="Pragma" content="no-cache">
    <meta http-equiv="Expires" content="0">
    </head>
    """

    with open(dest_path, "r", encoding="utf-8") as f:
        html_content = f.read()

    if "Cache-Control" not in html_content:
        html_content = html_content.replace("</head>", anti_cache_tags)

    if "background-color: #f6f8fa" not in html_content:
        html_content = re.sub(r'(<body[^>]*>)', r'\1\n' + header_banner, html_content, count=1, flags=re.IGNORECASE)

    with open(dest_path, "w", encoding="utf-8") as f:
        f.write(html_content)

    print("\n2. Pushing updates (HTML + Notebook) to GitHub...")
    html_rel_path = os.path.join("docs", HTML_FILENAME)

    try:
        subprocess.run(['git', 'add', html_rel_path, NOTEBOOK_REPO_REL_PATH], cwd=repo_root, check=True)
        subprocess.run(['git', 'commit', '-m', f'Auto-update HTML export and notebook source for {base_notebook_name} [{current_time}]'], cwd=repo_root, check=True)
        subprocess.run(['git', 'push', 'origin', 'main'], cwd=repo_root, check=True)
        print("Git push successful!")
    except subprocess.CalledProcessError as e:
        print(f"Git command failed: {e}")

    pages_url = f"https://{GITHUB_USERNAME}.github.io/{REPO_NAME}/{HTML_FILENAME}"
    print("\n--- DEPLOYMENT COMPLETE ---")
    print(f"Live GitHub Pages URL:\n{pages_url}")

In [ ]:
import os
import subprocess

repo_root = subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()
print(f"Scanning repository root: {repo_root}\n")

print("Found notebooks in your repository:")
for root, dirs, files in os.walk(repo_root):
    # Skip hidden folders like .git or .ipynb_checkpoints
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    for file in files:
        if file.endswith('.ipynb'):
            rel_path = os.path.relpath(os.path.join(root, file), repo_root)
            print(f"  - {rel_path}")